# Desafios Extras!

## Esses desafios são baseados em desafios reais para entrevistas de vagas para ciência de dados e machine learning


In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

# URL do arquivo cru no GitHub
url = "https://raw.githubusercontent.com/debs-b/desafio-uber/main/dados/dados_simulados_uber.csv"

# Baixa os dados diretamente do GitHub
dados = pd.read_csv(url)
print("Dados carregados do GitHub com sucesso!")

# Visualizando as primeiras linhas dos dados
dados.head()

In [ ]:
# Criando o DataFrame com as descrições traduzidas
colunas_descricao = {
    "Nome da Coluna": [
        "id", "city_id", "signup_os", "signup_channel", "signup_timestamp",
        "bgc_date", "vehicle_added_date", "first_trip_date", "vehicle_make",
        "vehicle_model", "vehicle_year"
    ],
    "Descrição": [
        "Identificador único do motorista.",
        "Identificador da cidade onde o usuário se cadastrou.",
        "Dispositivo utilizado para cadastro do usuário ('android', 'ios', 'website', 'other').",
        "Canal pelo qual o motorista se cadastrou ('offline', 'paid', 'organic', 'referral').",
        "Data e hora da criação da conta; horário local no formato ‘AAAA/MM/DD’.",
        "Data em que o consentimento para verificação de antecedentes foi dado; no formato ‘AAAA/MM/DD’.",
        "Data em que as informações do veículo do motorista foram carregadas; no formato ‘AAAA/MM/DD’.",
        "Data da primeira viagem como motorista; no formato ‘AAAA/MM/DD’.",
        "Marca do veículo cadastrado (ex.: Honda, Ford, Kia).",
        "Modelo do veículo cadastrado (ex.: Accord, Prius, 350z).",
        "Ano de fabricação do veículo; no formato ‘AAAA’."
    ]
}

descricao_df = pd.DataFrame(colunas_descricao)

In [ ]:
pd.set_option('display.max_columns', None)  # or 1000
pd.set_option('display.max_rows', None)  # or 1000
pd.set_option('display.max_colwidth', None)  # or 199

In [ ]:
# Exibindo o DataFrame
descricao_df

# Pergunta 1:
## Proponha e defina a métrica principal de sucesso do app redesenhado. Quais são 2-3 métricas adicionais de acompanhamento que serão importantes monitorar além da métrica principal definida?
## O foco principal é melhorar a experiência dos motoristas!

In [ ]:
# Verificar valores nulos em cada coluna
analise_nulos = dados.isnull().sum()

# Verificar a proporção de valores nulos
proporcao_nulos = (analise_nulos / len(dados)) * 100

# Consolidar em um DataFrame para facilitar a leitura
resumo_nulos = pd.DataFrame({
    'Coluna': dados.columns,
    'Valores Nulos': analise_nulos,
    'Proporção de Nulos (%)': proporcao_nulos
}).sort_values(by='Proporção de Nulos (%)', ascending=False)

resumo_nulos

In [ ]:
# só olhando para os nulos, o que você acha dessa análise inicial?

In [ ]:
# Análise

In [ ]:
# Possíveis métricas a serem criadas com base nos dados fornecidos
# Primeiro, vamos entender o context: Uber mudou seu app, e quer saber se isso afetou o engajamento de novos motoristas.

## O que já temos no dataset:

    Motoristas ativos:
    Podemos identificar motoristas ativos verificando a existência de uma data válida em first_completed_date, indicando que eles completaram pelo menos uma viagem.

    Canais e dispositivos de cadastro:
    Informações como signup_channel e signup_os podem ajudar a entender padrões de engajamento (ex.: motoristas que se cadastram via canal "referral" podem ter comportamentos diferentes).

    Tempo entre etapas:
    Calculando a diferença entre:
        signup_timestamp e first_completed_date (tempo para fazer a primeira viagem).
        signup_timestamp e vehicle_added_date (tempo para adicionar informações do veículo).
        bgc_date e first_completed_date (tempo desde a verificação de antecedentes até a primeira viagem).

    Ano do veículo (vehicle_year):
    Pode ser usado para derivar se motoristas com carros mais novos ou antigos têm maior engajamento.

Taxa de Motoristas Ativos:

Essa métrica indicará a proporção de motoristas que completaram pelo menos uma viagem.

Lógica: Motoristas podem se cadastrar mas nunca fazer sua primeira viagem. A proporção de motoristas que fizeram sua primeira viagem (ativos) / total de motoristas é a métrica que estamos interessados como métrica principal.

In [ ]:
# Convertendo 'first_completed_date' para datetime
dados['first_trip_date'] = pd.to_datetime(dados['first_completed_date'], errors='coerce')

# Identificando motoristas ativos (que possuem uma data de primeira viagem)
motoristas_ativos = dados[dados['first_completed_date'].notnull()]

# Calculando a taxa de motoristas ativos
taxa_motoristas_ativos = (len(motoristas_ativos) / len(dados)) * 100

print(f"Taxa de Motoristas Ativos: {taxa_motoristas_ativos:.2f}%")


## Metrica: Taxa de Ativacao de Novos Motoristas

# O que você acha desse valor?
## Quais possíveis hipóteses justificariam esse valor?

Outras métricas ajudam a entender qual o grande desafio para os motoristas serem ativos.

## Sub metrica: Taxa de Motoristas que adicionaram informacoes do veiculo

In [ ]:
# Taxa de Motoristas que Adicionaram o Veículo

# Convertendo 'vehicle_added_date' para datetime
dados['vehicle_added_date'] = pd.to_datetime(dados['vehicle_added_date'], errors='coerce')

# Calculando a taxa de motoristas que adicionaram informações do veículo
motoristas_com_veiculo = dados[dados['vehicle_added_date'].notnull()]
taxa_motoristas_veiculo = (len(motoristas_com_veiculo) / len(dados)) * 100

print(f"Taxa de Motoristas que adicionaram informações do veículo: {taxa_motoristas_veiculo:.2f}%")


## Sub Metrica: Taxa de Motoristas que Passaram pela Verificação de Antecedentes

In [ ]:
# Taxa de Motoristas que Passaram pela Verificação de Antecedentes
# Convertendo 'bgc_date' para datetime
dados['bgc_date'] = pd.to_datetime(dados['bgc_date'], errors='coerce')

# Calculando a taxa de motoristas que passaram pela verificação de antecedentes
motoristas_bgc = dados[dados['bgc_date'].notnull()]
taxa_motoristas_bgc = (len(motoristas_bgc) / len(dados)) * 100

print(f"Taxa de Motoristas que completaram a verificação de antecedentes: {taxa_motoristas_bgc:.2f}%")


## Sub Metricas de tempo: Tempo médio entre cadastro e envio de informações do veículo e Tempo médio entre cadastro e primeira viagem

In [ ]:
# Tempo medio entre etapas
# Calculando o tempo médio entre etapas (em dias)
tempo_cadastro_para_veiculo = (dados['vehicle_added_date'] - pd.to_datetime(dados['signup_date'])).dt.days
tempo_cadastro_para_viagem = (dados['first_trip_date'] - pd.to_datetime(dados['signup_date'])).dt.days

# Excluindo valores nulos
tempo_cadastro_para_veiculo_medio = tempo_cadastro_para_veiculo.dropna().mean()
tempo_cadastro_para_viagem_medio = tempo_cadastro_para_viagem.dropna().mean()

print(f"Tempo médio entre cadastro e envio de informações do veículo: {tempo_cadastro_para_veiculo_medio:.2f} dias")
print(f"Tempo médio entre cadastro e primeira viagem: {tempo_cadastro_para_viagem_medio:.2f} dias")


In [ ]:
# Taxas por canal de cadastro
# Calculando a taxa de motoristas ativos por canal de cadastro
taxas_por_canal = dados.groupby('signup_channel')['first_trip_date'].apply(lambda x: x.notnull().mean() * 100)

print("Taxas de motoristas ativos por canal de cadastro:")
print(taxas_por_canal)


In [ ]:
# Compilação de métricas

# Consolidando as métricas calculadas
metricas = {
    "Métrica": [
        "Taxa de Motoristas Ativos",
        "Taxa de Motoristas que Adicionaram o Veículo",
        "Taxa de Motoristas que Passaram pela Verificação de Antecedentes",
        "Tempo Médio entre Cadastro e Envio de Informações do Veículo",
        "Tempo Médio entre Cadastro e Primeira Viagem",
        "Taxa de Motoristas Ativos por Canal (Referrals)",
        "Taxa de Motoristas Ativos por Canal (Paid)",
        "Taxa de Motoristas Ativos por Canal (Organic)"
    ],
    "Valor": [
        taxa_motoristas_ativos, 
        taxa_motoristas_veiculo,  
        taxa_motoristas_bgc,  
        tempo_cadastro_para_veiculo_medio,  
        tempo_cadastro_para_viagem_medio, 
        taxas_por_canal["Organic"],
        taxas_por_canal["Paid"],
        taxas_por_canal["Referral"]   
    ],
    "Descrição": [
        "Proporção de motoristas cadastrados que completaram pelo menos uma viagem.",
        "Proporção de motoristas que adicionaram informações do veículo após o cadastro.",
        "Proporção de motoristas que completaram a verificação de antecedentes.",
        "Tempo médio desde o cadastro até o envio das informações do veículo.",
        "Tempo médio desde o cadastro até a realização da primeira viagem.",
        "Proporção de motoristas ativos que vieram pelo canal 'referral'.",
        "Proporção de motoristas ativos que vieram pelo canal 'paid'.",
        "Proporção de motoristas ativos que vieram pelo canal 'organic'."
    ]
}

# Criando o DataFrame
df_metricas = pd.DataFrame(metricas)

# Exibindo o DataFrame
df_metricas


# Quais conclusões podemos tirar dessas métricas?

## manter o foco! Um grande problema de candidatos é perder o foco em exercícios de ciência de dados. Qual era a pergunta:
## Proponha e defina a métrica principal de sucesso do app redesenhado. Quais são 2-3 métricas adicionais de acompanhamento que serão importantes monitorar além da métrica principal definida?


## Resposta: A métrica principal é a Taxa de Motoristas Ativos, que calcula a proporção entre motoristas ativos (que fizeram sua primeira viagem) e total de motoristas cadastrados. Esse valor é 12.81%, uma baixo valor. Outras métricas para analisar onde está o "gargalo" entre cadatro e ativação são:
"Taxa de Motoristas Ativos",
"Taxa de Motoristas que Adicionaram o Veículo",
"Taxa de Motoristas que Passaram pela Verificação de Antecedentes",
"Tempo Médio entre Cadastro e Envio de Informações do Veículo",
"Tempo Médio entre Cadastro e Primeira Viagem",
"Taxa de Motoristas Ativos por Canal (Referrals)",
"Taxa de Motoristas Ativos por Canal (Paid)",
"Taxa de Motoristas Ativos por Canal (Organic)"

E seus valores estão na tabela acima. Baseado na tabela, a gende maioria dos candidatos, 64.99%, passa pela checagem de antecedentes, então esse processo não parace ser o que diminuiu o engajamento. Apenas 28.99% adicionam o seus veículos, mostrando que talvez o processo de adicionar as informações do veículo seja complicado ou de dificíl engajamento. 
Sobre os canais de engajamento, organic é o canal com maior taxa de motoristas ativos. Referências tem um engajamento alto de 19%, mostrando sucesso dos programas de referência e possíveis maiores investimentos nos mesmos.

Percebeu que foi pouco código para muito insight? Isso é ciência de dados! Há momentos de só ficar no código, e momentos de analisar os resultados e trazer ganhos ao negócio!

### Resumo insights: Olhando para os dados brutos, a maioria dos candidatos foca em código. O cientista de dados sênior foca no padrão: 64.99% dos motoristas passam na checagem de antecedentes (BGC), mas apenas 28.99% adicionam um veículo.
#### O gargalo não é a burocracia do governo (antecedentes), o gargalo é a etapa de cadastro do veículo dentro do aplicativo. O novo design precisa atacar exatamente essa tela/fluxo.

# Pergunta 2: Descreva um plano de teste para avaliar se o app redesenhado apresenta melhor desempenho (de acordo com as métricas que você definiu). Como você equilibraria a necessidade de obter resultados rápidos, com rigor estatístico e ainda monitorando riscos?

## Resposta: Teste A/B

## O Desenho do Experimento (Split de Usuários):

   * Unidade de Randomização: O motorista (baseado no id). Como o marketplace da Uber é dinâmico, randomizar por motorista em uma mesma cidade pode gerar efeito de rede (network effects / interferência).

   * Idealmante o correto seria um Cluster-Based A/B Test (ex: dividir por cidades inteiras ou regiões bem definidas).
   
   * Para simplificar o código, assumiremos um split clássico por usuário, mas trazendo esse alerta de negócio.

## Dimensionamento Estatístico (Sample Size & MDE):
* Métrica Principal: Taxa de Ativação (Base atual: ~12.81%).

* MDE (Efeito Mínimo Detectável): Definir o quanto de melhoria queremos detectar que faça sentido financeiro (ex: aumentar a ativação em 2 pontos percentuais, indo para 13.22%).

* Poder Estatístico ($1 - \beta$): 80%.

* Significância ($\alpha$): 5% (p-valor < 0.05).

* Duração: O teste deve rodar por ciclos completos de semanas (mínimo 2 semanas, idealmente 4 semanas devido ao ciclo de ativação de 13 dias do motorista) para evitar sazonalidades de dias de semana vs. finais de semana.

## Métricas de Guardrail:

* O que monitorar para garantir que o novo app não quebrou o negócio? Taxa de Cancelamento de Viagens e Tempo de Espera do Passageiro (ETA). Se a ativação subir, mas os cancelamentos dispararem porque a nova interface confunde o motorista, o teste deve ser abortado.

* Taxa de Motoristas que Adicionaram o Veículo: Verificar se o redesenho simplifica o processo de adicionar informações do veículo.

* Tempo Médio entre Cadastro e Envio de Informações do Veículo: Avaliar se o redesenho reduz o tempo necessário para essa etapa.

* Tempo Médio entre Cadastro e Primeira Viagem: Identificar se o redesenho agiliza a ativação.

### Riscos a Monitorar:

* Feedback negativo: Analisar comentários e avaliações para identificar possíveis problemas ou insatisfações com o app redesenhado.

* Retenção de longo prazo: Garantir que motoristas do grupo tratamento permaneçam engajados após a ativação inicial.

In [ ]:

# Objetivo: Definir o tamanho da amostra (Sample Size) e o tempo de rodada do 
# Teste A/B para garantir que o resultado não seja fruto do acaso (p-hacking).

import statsmodels.stats.api as sms
import numpy as np

print("--- CONFIGURAÇÃO DO EXPERIMENTO ESTATÍSTICO ---")

# 1. Parâmetros de Entrada (Métricas de Negócio da Uber)
# Usamos a taxa atual de ativação que calculamos no Passo 2 como nossa linha de base (Baseline)
taxa_controle_baseline = taxa_motoristas_ativos / 100  # 0.1122 (ou 12.81%)

# MDE (Minimum Detectable Effect): O menor efeito que o time de produto julga 
# relevante para justificar o custo do redesenho. Vamos testar um aumento de 2 pontos percentuais.
mde = 0.02 
taxa_tratamento_esperada = taxa_controle_baseline + mde

# Parâmetros Estatísticos Padrão de Mercado
alpha = 0.05  # Nível de Significância (Chance de falso positivo tolerada: 5%)
power = 0.80  # Poder Estatístico (Chance de detectar um efeito real: 80%)

print(f"Conversão Atual (Controle): {taxa_controle_baseline*100:.2f}%")
# Renderizando equações simples em markdown limpo (sem LaTeX desnecessário)
print(f"Conversão Alvo (Tratamento): {taxa_tratamento_esperada*100:.2f}%")
print(f"Efeito Mínimo Detectável (MDE): {mde*100:.2f}%\n")

# 2. Cálculo do Tamanho de Amostra Necessário
# Calculamos o 'Effect Size' usando a transformação de Cohen (Cohen's h) para proporções
effect_size = sms.proportion_effectsize(taxa_controle_baseline, taxa_tratamento_esperada)

# Resolução do poder estatístico para encontrar o N (tamanho da amostra por grupo)
analise_poder = sms.NormalIndPower()
tamanho_amostra_grupo = analise_poder.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0 # Divisão 50/50 entre Grupo Controle e Grupo Tratamento
)

tamanho_amostra_grupo = int(np.ceil(tamanho_amostra_grupo))
amostra_total = tamanho_amostra_grupo * 2

print("--- RESULTADO DO DIMENSIONAMENTO ---")
print(f"Número de motoristas necessários no Grupo Controle: {tamanho_amostra_grupo:,}")
print(f"Número de motoristas necessários no Grupo Tratamento: {tamanho_amostra_grupo:,}")
print(f"Volume total de novos cadastros necessários: {amostra_total:,}\n")

# 3. Estimativa de Tempo de Execução do Teste
# Sabendo quantos novos motoristas se cadastram, estimamos a duração do teste.
novos_cadastros_totais = len(dados) # Total de cadastros no nosso histórico
print(f"O dataset completo possui {novos_cadastros_totais:,} cadastros históricos.")

# Supondo que esse dataset represente o tráfego acumulado de, por exemplo, 4 semanas (28 dias):
print("\n--- DECISÃO DE PRODUTO & MITIGAÇÃO DE RISCO ---")
print("1. Duração do Teste: O teste deve rodar por ciclos semanais completos (mínimo 14 dias)")
print("   para capturar o comportamento dos motoristas tanto em dias úteis quanto em fins de semana.")
print("2. Mitigação de Risco: O rollout inicial deve ser gradual (ex: 90/10) para garantir estabilidade.")
print("3. Métricas de Guarda-Chuva: Se a Taxa de Cancelamento ou o Tempo de Espera (ETA) subirem, e outras métricas de experiência do usuário piorarem,")
print("   o experimento deve ser abortado imediatamente, mesmo que a ativação esteja subindo.")

## Notas:

* Effect Size: Para detectar variações pequenas (como $2\%$), o volume de dados necessário cresce bastante. Se quiséssemos detectar um aumento de apenas $0.5\%$, o tamanho da amostra seria brutalmente maior.

In [ ]:
tamanho_amostra_grupo = analise_poder.solve_power(
    effect_size=0.005,
    alpha=alpha,
    power=power,
    ratio=1.0 # Divisão 50/50 entre Grupo Controle e Grupo Tratamento
)

tamanho_amostra_grupo = int(np.ceil(tamanho_amostra_grupo))
amostra_total = tamanho_amostra_grupo * 2

print("--- DIMENSIONAMENTO PARA DETECTAR 0.5% ---")
print(f"Número de motoristas necessários no Grupo Controle: {tamanho_amostra_grupo:,}")
print(f"Número de motoristas necessários no Grupo Tratamento: {tamanho_amostra_grupo:,}")
print(f"Volume total de novos cadastros necessários: {amostra_total:,}\n")

* Métricas Guardrail: métricas puras de conversão podem ser enganosas se destruírem o ecossistema do marketplace (ex: ativar mais motoristas, mas que aceitam corridas e cancelam logo em seguida).

# Pergunta 3: Explique como você traduziria os resultados do plano de teste em uma decisão sobre lançar o novo design ou revertê-lo.

## 1. Cenário Ideal (Greenlight)
* Condição: Aumento na Métrica Principal (Taxa de Ativação) é estatisticamente significante ($p < 0.05$) E o ganho é maior ou igual ao MDE definido (ex: $+2\%$ de ativação).

* Métricas Guardrail: Nenhuma métrica de guardrail (como cancelamentos ou reclamações no suporte) piorou de forma significante.

* Decisão: Lançar o novo design via Rollout Gradual (ex: expandir de 10% para 25%, 50% e 100% ao longo de duas semanas para monitorar a carga dos servidores e suporte).

## 2. Cenário a ser Discutido (O Verdadeiro Desafio do Cientista de Dados)
* Condição A (Efeito Estatístico sem Impacto de Negócio): O resultado é estatisticamente significante ($p < 0.05$), mas o aumento real foi de apenas $0.1\%$.

* Análise: O teste teve muita amostra, mas o ganho não paga o custo de desenvolvimento e treinamento dos motoristas. 

* Decisão: *Reverter* ou *Iterar* (volta ao design e novo teste).



---

* Condição B (Métricas Conflitantes): A taxa de ativação subiu $3\%$, mas a taxa de cancelamento de viagens também subiu $5\%$.

* Análise: O novo app facilita o cadastro, mas confunde o motorista na hora de fazer a corrida. 

* Decisão: Abortar o lançamento e devolver para o time de UX/Design.

## 3. Cenário de Falha (Reversão)

* Condição: Sem significância estatística ($p > 0.05$) ou resultado negativo na ativação.

* Decisão: Reverter imediatamente. O design antigo permanece como o campeão (Champion).

### Notas:

* Métrica não é tudo: O novo app pode ser um sucesso para "cadastrar mais rápido", mas um fracasso para o negócio se o motorista começar a cancelar mais viagens por não entender a nova tela.


* Custo de Mudança (Switching Cost): Mudar o aplicativo de milhões de usuários exige custo de servidores, atualização de FAQs, suporte técnico e atrito de aprendizado do usuário. 
* Por isso, se o ganho estatístico for real, mas muito pequeno, a decisão correta de negócio costuma ser não lançar.


## Parte 2: Realize a limpeza de dados necessária, análise exploratória e/ou visualizações utilizando o dataset fornecido (descrições breves ou gráficos que ilustrem sua abordagem são incentivados). Qual a porcentagem de motoristas cadastrados que resultaram em uma primeira viagem?

In [ ]:

# Objetivo: Realizar a limpeza dos dados, converter tipos de variáveis, 
# responder à pergunta de conversão e gerar gráficos.

import matplotlib.pyplot as plt
import seaborn as sns

print("--- ETAPA 1: TRATAMENTO E TIPAGEM DOS DADOS ---")

# As colunas de data no arquivo original vêm como texto (string/object).
# Precisamos convertê-las para datetime. O parâmetro errors='coerce' garante 
# que valores inválidos ou nulos (NaN) não quebrem o código.
colunas_data = ['signup_date', 'bgc_date', 'vehicle_added_date', 'first_completed_date']

for coluna in colunas_data:
    if coluna in dados.columns:
        dados[coluna] = pd.to_datetime(dados[coluna], errors='coerce')

# Ajustando o nome da coluna de timestamp do cadastro para manter o padrão do dicionário
if 'signup_date' in dados.columns:
    dados = dados.rename(columns={'signup_date': 'signup_timestamp'})

print("Tipagem das colunas atualizada com sucesso!\n")


print("--- ETAPA 2: CÁLCULO DA TAXA DE CONVERSÃO FINAL ---")

# Motoristas ativos são aqueles que possuem uma data válida em 'first_completed_date'
total_cadastros = len(dados)
motoristas_ativos = dados[dados['first_completed_date'].notnull()]
total_ativos = len(motoristas_ativos)

taxa_conversao_final = (total_ativos / total_cadastros) * 100

print(f"Total de motoristas que iniciaram o cadastro: {total_cadastros:,}")
print(f"Total de motoristas que realizaram a primeira viagem: {total_ativos:,}")
print(f"PORCENTAGEM DE MOTORISTAS ATIVADOS: {taxa_conversao_final:.2f}%\n")


print("--- ETAPA 3: GERANDO VISUALIZAÇÕES E INSIGHTS PARA AULA ---")

# Configurando o estilo visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11})

# Gráfico 1: O Funil de Conversão Comercial (Onde está o vazamento de receita?)
etapas_funil = [
    '1. Cadastros Totais', 
    '2. Passaram Antecedentes (BGC)', 
    '3. Adicionaram Veículo', 
    '4. Ativados (1ª Viagem)'
]
valores_funil = [
    total_cadastros,
    dados['bgc_date'].notnull().sum(),
    dados['vehicle_added_date'].notnull().sum(),
    total_ativos
]

plt.figure(figsize=(10, 5))
barplot = sns.barplot(x=valores_funil, y=etapas_funil, palette="Blues_r", edgecolor="black")
plt.title('Funil de Ativação de Novos Motoristas Uber', fontsize=14, pad=15)
plt.xlabel('Quantidade de Motoristas')

# Adicionando rótulos com as porcentagens em relação ao total de cadastros
for i, valor in enumerate(valores_funil):
    porcentagem = (valor / total_cadastros) * 100
    plt.text(valor + (total_cadastros * 0.01), i, f"{valor:,} ({porcentagem:.1f}%)", va='center', fontweight='bold')

plt.xlim(0, total_cadastros * 1.2)
plt.tight_layout()
plt.show()


# Gráfico 2: Comportamento de Ativação por Canal de Aquisição (Signup Channel)
# Queremos saber se canais orgânicos ou indicações ativam melhor que mídia paga.
dados['ativado'] = dados['first_completed_date'].notnull()
taxa_por_canal = dados.groupby('signup_channel')['ativado'].mean() * 100
taxa_por_canal = taxa_por_canal.sort_values(ascending=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=taxa_por_canal.index, y=taxa_por_canal.values, palette="viridis", edgecolor="black")
plt.title('Taxa de Ativação (%) por Canal de Cadastro', fontsize=14, pad=15)
plt.ylabel('Taxa de Conversão (%)')
plt.xlabel('Canal de Origem do Motorista')

for i, valor in enumerate(taxa_por_canal.values):
    plt.text(i, valor + 0.5, f"{valor:.1f}%", ha='center', va='bottom', fontweight='bold')

plt.ylim(0, taxa_por_canal.max() * 1.15)
plt.tight_layout()
plt.show()


# Gráfico 3: Distribuição do Tempo de Ativação (Time-to-First-Trip)
# Filtramos apenas os motoristas que de fato completaram uma viagem para ver o delay do processo.
dados_ativos = dados[dados['first_completed_date'].notnull()].copy()
dados_ativos['dias_para_ativar'] = (dados_ativos['first_completed_date'] - dados_ativos['signup_timestamp']).dt.days

plt.figure(figsize=(9, 4.5))
sns.histplot(data=dados_ativos, x='dias_para_ativar', bins=30, kde=True, color="purple")
plt.title('Distribuição de Dias até a Primeira Viagem (Apenas Ativados)', fontsize=14, pad=15)
plt.xlabel('Dias decorridos entre Cadastro e Ativação')
plt.ylabel('Frequência de Motoristas')
plt.axvline(dados_ativos['dias_para_ativar'].mean(), color='red', linestyle='--', label=f"Média: {dados_ativos['dias_para_ativar'].mean():.1f} dias")
plt.legend()
plt.tight_layout()
plt.show()

## Resposta: A porcentagem de motoristas cadastrados que resultaram em uma primeira viagem é de exatamente 12.81%.
( sempre escreva respostas claras e diretas) - comentario da experiencia

### Grafico 2: 
#### O poder de negócios desse insight. O canal Referral (Indicação) tem quase o dobro de eficiência de conversão ($19.9\%$) em relação ao tráfego Paid (Mídia Paga - $6.2\%$ ). 
#### Motoristas indicados por outros motoristas são muito mais propensos a passar pelas barreiras do app.

### O Delay de Engajamento (Gráfico 3):
#### A média de tempo para fazer a primeira corrida é de 13 dias. Isso prova que o processo de ativação não é imediato;
#### exige um acompanhamento de notificações por push, e-mails de incentivo, etc ao longo das duas primeiras semanas do usuário.

In [ ]:
# Para responder a primeira pergunta, nós já havíamos feito várias análises. Isso é comum em desafios: as vezes a dica para pergunta que você está com dúvidas, está na próxima pergunta! 
# Então leia todas as perguntas antes de começar seu exercício.
# Mantenha códico claro e preciso. Mantenha o foco e sempre responda a todas as perguntas. Se sobrar tempo, adicione comentários extras. Seja pro-ativo(a) e criativo(a)! Não cole. LLMs pode te ajudar a estudar, mas na hora do desafio ao vivo, mantenha-se firme em seus conhecimentos. 

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
len(dados)

In [ ]:
# Convertendo colunas para formato datetime
dados['signup_timestamp'] = pd.to_datetime(dados.get('signup_date', dados.get('signup_timestamp')), errors='coerce')
dados['bgc_date'] = pd.to_datetime(dados['bgc_date'], errors='coerce')
dados['vehicle_added_date'] = pd.to_datetime(dados['vehicle_added_date'], errors='coerce')
dados['first_trip_date'] = pd.to_datetime(dados.get('first_trip_date', dados.get('first_completed_date')), errors='coerce')


In [ ]:
# Motoristas que completaram a primeira viagem
motoristas_ativos = dados[dados['first_trip_date'].notnull()]

taxa_motoristas_ativos = (len(motoristas_ativos) / len(dados)) * 100
print(f"Porcentagem de motoristas que fizeram a primeira viagem: {taxa_motoristas_ativos:.2f}%")


In [ ]:
# Dados para o gráfico
status = ['Ativos (Primeira Viagem)', 'Inativos (Sem Primeira Viagem)']
valores = [len(motoristas_ativos), len(dados) - len(motoristas_ativos)]

# Criando o gráfico
plt.figure(figsize=(8, 5))
plt.bar(status, valores, color=['green', 'red'])
plt.title('Distribuição de Motoristas Ativos e Inativos', fontsize=14)
plt.ylabel('Número de Motoristas', fontsize=12)
plt.show()


In [ ]:
# Visualização 1: Proporção de motoristas ativos e inativos
status = ['Ativos (Primeira Viagem)', 'Inativos (Sem Primeira Viagem)']
valores = [len(motoristas_ativos), len(dados) - len(motoristas_ativos)]

plt.figure(figsize=(8, 5))
plt.pie(valores, labels=status, autopct='%1.1f%%', colors=['green', 'red'], startangle=90)
plt.title('Proporção de Motoristas Ativos e Inativos', fontsize=14)
plt.show()

# Visualização 2: Distribuição de motoristas por canal de cadastro (signup_channel)
canal_ativacao = dados.groupby('signup_channel')['first_trip_date'].apply(lambda x: x.notnull().mean() * 100)
plt.figure(figsize=(8, 5))
canal_ativacao.sort_values().plot(kind='bar', color='blue', edgecolor='black')
plt.title('Taxa de Ativação por Canal de Cadastro', fontsize=14)
plt.ylabel('Taxa de Ativação (%)', fontsize=12)
plt.xlabel('Canal de Cadastro', fontsize=12)
plt.xticks(rotation=45)
plt.show()

# Visualização 3: Distribuição do tempo entre cadastro e primeira viagem (dias)
dados['tempo_primeira_viagem'] = (dados['first_trip_date'] - dados['signup_timestamp']).dt.days
dados['tempo_primeira_viagem'] = dados['tempo_primeira_viagem'].dropna()

plt.figure(figsize=(8, 5))
dados['tempo_primeira_viagem'].hist(bins=30, color='purple', edgecolor='black')
plt.title('Distribuição do Tempo entre Cadastro e Primeira Viagem', fontsize=14)
plt.xlabel('Dias até a Primeira Viagem', fontsize=12)
plt.ylabel('Número de Motoristas', fontsize=12)
plt.show()